# Colombia Open — Municipios

Este notebook muestra cómo cargar el dataset **`municipios`** (DIVIPOLA, DANE) con `colombia-open`: código de 5 dígitos, nombre oficial, departamento, centroide, y cómo unirlo con `departamentos` y resolver nombres de ciudades.

> **Nota (2026-10):** el proyecto todavía no está publicado en PyPI ni tiene un Release con datos pre-construidos (ver `README.md` del repo, sección "Estado del proyecto"). Por eso este notebook clona el repo e instala el extra `pipeline` para generar los datos localmente en vez de hacer `pip install colombia-open`. Cuando exista un Release publicado, la celda de setup se reduce a `pip install colombia-open`.

## Setup

In [ ]:
!git clone --depth 1 https://github.com/davidcasr/colombia-open.git
%cd colombia-open
!pip install -q -e ".[pipeline]"
!python -m src.build_pipeline

## Cargar el dataset

In [ ]:
from colombia_open import ColombiaOpen

hub = ColombiaOpen()
municipios = hub.load_polars("municipios")

print(municipios.shape)
municipios.head(10)

## Esquema

| Columna | Tipo | Descripción |
|---|---|---|
| `cod_dpto` | string (2 dígitos) | Join key hacia `departamentos`. |
| `nombre_dpto` | string | Nombre del departamento, denormalizado para evitar un join obligatorio. |
| `cod_mpio` | string (5 dígitos) | Código DIVIPOLA de municipio. **Nunca se convierte a int.** |
| `nombre_mpio` | string | Nombre oficial del municipio (puede diferir del nombre coloquial, ej. Cali es "SANTIAGO DE CALI"). |
| `nombre_mpio_clean` | string | `nombre_mpio` normalizado — usado por `hub.resolve_municipios()`. |
| `tipo_municipio` | string | "Municipio", "Capital", u otra categoría según la fuente. |
| `longitud` / `latitud` | float | Centroide en grados decimales. Puede ser nulo. |

## Filtrar municipios de un departamento

In [ ]:
# Municipios de Antioquia (cod_dpto == "05")
municipios.filter(municipios["cod_dpto"] == "05").select(
    ["cod_mpio", "nombre_mpio", "tipo_municipio"]
).head(10)

## Resolver nombres de ciudades

`resolve_municipios()` hace coincidencia exacta tras normalizar (sin tildes, minúsculas) — no es fuzzy matching. Usa el nombre oficial DIVIPOLA o el código si lo conoces.

In [ ]:
hub.resolve_municipios(["Medellín", "Santiago de Cali"])

## Join con `departamentos`

In [ ]:
hub.cross_view(["municipios", "departamentos"], on="cod_dpto").head(10)

## Limitaciones conocidas

- Sin geometría de polígonos (MGN) todavía — solo código + nombre + centroide.
- `resolve_municipios()` no hace fuzzy matching: "Cali" no resuelve porque el nombre oficial es "SANTIAGO DE CALI".

Documentación completa del esquema: [`docs/datasets/municipios.md`](../../docs/datasets/municipios.md).